# Accessing Climate Model Projections in Python

This notebook is for researchers who know R and want to start using Python for climate data analysis. R has excellent climate tools (`terra`, `stars`, `tidync`), so this isn't a claim that Python is better overall — but for **accessing large CMIP6 archives from cloud storage**, Python has a meaningful advantage.

The key tool is `intake-esm`, which has no R equivalent. It provides a searchable catalog of CMIP6 data hosted on AWS S3 and lets you load model output directly into memory without downloading files first. The traditional alternative — fetching data from ESGF (the Earth System Grid Federation) — requires creating an account, submitting data requests, and waiting for large file transfers. `intake-esm` skips all of that.

**What you'll learn:**
1. How to search the CMIP6 data catalog using `intake-esm`
2. How to load and explore climate model output using `xarray`
3. How to subset data to a region and make a time series plot

**Region of interest:** The Great Lakes Basin (Canada/US)

**Variable:** Near-surface air temperature (`tas`), comparing a historical run to two future emissions scenarios (SSP2-4.5 and SSP5-8.5)

## What is CMIP6?

CMIP6 (Coupled Model Intercomparison Project Phase 6) is a coordinated global effort in which roughly 50 modeling groups ran their climate models under the same experimental protocols. The result is a large, standardized archive of climate simulations that researchers use to understand past climate and project future change.

The archive is organized around a few key concepts:

- **Models** (`source_id`): Different institutions' implementations of Earth system physics — atmosphere, ocean, land, sea ice. Examples: CESM2 (US/NCAR), CanESM5 (Canada), MIROC6 (Japan).
- **Experiments** (`experiment_id`): The `historical` experiment covers 1850–2014, forced with observed greenhouse gases. Future projections use *Shared Socioeconomic Pathways* (SSPs) — `ssp245` is a moderate-emissions future, `ssp585` is high emissions.
- **Variables** (`variable_id`): Standard outputs like `tas` (near-surface air temperature), `pr` (precipitation), `psl` (sea-level pressure), and hundreds more.
- **Ensemble members** (`member_id`): Each model may be run multiple times with slightly different starting conditions (e.g., `r1i1p1f1`, `r2i1p1f1`) to sample internal variability.

All CMIP6 data is freely available — this notebook accesses it directly from AWS S3, so there's nothing to download in advance.

## 1. Setting Up Your Environment

In R you use `install.packages()`. In Python, we manage packages with **conda** or **mamba** (mamba is a faster drop-in replacement for conda).

This repo includes an `environment.yml` file that defines all required packages. Run this **once in your terminal** to create the environment:

```bash
# with mamba (recommended — much faster)
mamba env create -f environment.yml

# or with conda
conda env create -f environment.yml
```

Then activate it and launch Jupyter:

```bash
conda activate cmip6
jupyter notebook
```

You only need to create the environment once. After that, just activate it each time.

## 2. Import Packages

The `import` statement is Python's equivalent of R's `library()`. The `as` keyword creates a short alias — you'll see `xr`, `np`, and `pd` used everywhere in the scientific Python world.

In [ ]:
import intake          # access data catalogs (like a smart file index)
import xarray as xr   # multidimensional climate data (like terra/raster in R)
import numpy as np    # numerical arrays (like base R vectors/matrices)
import pandas as pd   # tabular data (like data.frames in R)
import matplotlib.pyplot as plt  # plotting (like a lower-level ggplot2)

ModuleNotFoundError: No module named 'intake'

## 3. Open the CMIP6 Catalog

CMIP6 data lives on AWS S3. Rather than navigating thousands of files directly, `intake-esm` gives us a searchable **catalog** — think of it as a master table that maps model/variable/scenario combinations to their S3 file paths.

Opening the catalog does **not** download any climate data. It only loads the index.

In [ ]:
catalog = intake.open_esm_datastore(
    'https://cmip6-pds.s3.amazonaws.com/pangeo-cmip6.json'
)

print(catalog)

## 4. Search the Catalog

`catalog.search()` filters the catalog — it's like `dplyr::filter()` in R. The key search terms are:

| Parameter | Meaning | Our choice |
|---|---|---|
| `source_id` | Climate model | CESM2 (US, from NCAR) |
| `experiment_id` | Scenario | historical, ssp245, ssp585 |
| `variable_id` | Climate variable | `tas` = near-surface air temperature |
| `table_id` | Time resolution & realm | `Amon` = monthly atmosphere |
| `member_id` | Ensemble member | `r10i1p1f1` |

The **historical** experiment covers 1850–2014. **SSP2-4.5** and **SSP5-8.5** are future projections (2015–2100) under moderate and high emissions, respectively.

In [ ]:
results = catalog.search(
    source_id='CESM2',
    experiment_id=['historical', 'ssp245', 'ssp585'],
    variable_id='tas',
    table_id='Amon',
    member_id='r10i1p1f1'
)

# .df gives us a pandas DataFrame — like a data.frame in R
results.df[['source_id', 'experiment_id', 'member_id', 'variable_id', 'zstore']]

## 5. Load the Data

The `zstore` column in the results table holds the S3 path to each dataset. We use `xr.open_zarr()` to open these files.

**Important:** `open_zarr()` uses **lazy loading** — it reads the metadata immediately but doesn't download the full data until you actually need the values. This is similar to how `arrow` works in R. The actual download happens later when we call `.compute()`.

We filter the results DataFrame to get the right S3 path for each scenario.

In [ ]:
# Filter the results table to get the S3 path for each experiment
# This is like results.df %>% filter(experiment_id == 'historical') %>% pull(zstore) in R
def get_path(df, experiment):
    return df.loc[df['experiment_id'] == experiment, 'zstore'].values[0]

path_hist  = get_path(results.df, 'historical')
path_ssp245 = get_path(results.df, 'ssp245')
path_ssp585 = get_path(results.df, 'ssp585')

# Open each dataset lazily from S3 (no data downloaded yet)
# storage_options={'anon': True} means public access — no AWS credentials needed
ds_hist   = xr.open_zarr(path_hist,   storage_options={'anon': True})
ds_ssp245 = xr.open_zarr(path_ssp245, storage_options={'anon': True})
ds_ssp585 = xr.open_zarr(path_ssp585, storage_options={'anon': True})

# Preview the historical dataset structure
print(ds_hist)

## 6. Understanding xarray Datasets

`xarray` is the backbone of scientific Python — it handles labeled, multi-dimensional data. If you've used R's `terra` or `raster`, the concepts are similar.

An `xr.Dataset` has:
- **Dimensions**: the named axes (`time`, `lat`, `lon`)
- **Coordinates**: the actual values along each axis (dates, degrees)
- **Data variables**: the arrays of data (here, just `tas`)

Quick R-to-Python translation:

```
dim(r)                 →  ds.dims  or  da.shape
r['tas']               →  ds['tas']  or  ds.tas
r[r$lat > 40, ]        →  ds.sel(lat=slice(40, 51))
r[1, ]                 →  ds.isel(time=0)
apply(r, c(2,3), mean) →  ds.mean(dim='time')
```

In [ ]:
# Access the 'tas' variable — this is an xr.DataArray (like a single raster layer)
tas = ds_hist['tas']

print("Dimensions:", dict(tas.dims))   # like dim() in R — but named
print("Shape:     ", tas.shape)         # like dim() in R — just the numbers
print("Units:     ", tas.attrs.get('units', 'not specified'))
print()
print("Time range:", str(tas.time.values[0])[:10], "to", str(tas.time.values[-1])[:10])
print("Lat range: ", float(tas.lat.min()), "to", float(tas.lat.max()), "°N")
print("Lon range: ", float(tas.lon.min()), "to", float(tas.lon.max()), "°E (0–360 convention)")

## 7. Subset to the Great Lakes Region

CMIP6 data uses **0–360° longitude** (not –180 to 180). The Great Lakes region sits at roughly 263–290°E, 39–51°N.

`.sel(lat=slice(39, 51))` selects by coordinate value — equivalent to filtering by value in R. `slice()` here defines a range (note: this is Python's `xarray.slice`, not base R's `slice()`).

In [ ]:
# Great Lakes bounding box
lat_min, lat_max = 39.0, 51.0
lon_min, lon_max = 263.0, 290.0  # 0-360° convention

# Subset each dataset to the region
gl_hist   = ds_hist.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
gl_ssp245 = ds_ssp245.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
gl_ssp585 = ds_ssp585.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))

print("Subset shape (time, lat, lon):", gl_hist['tas'].shape)

## 8. Compute Spatial Mean

We want a single time series representing the average temperature across the Great Lakes region. `.mean(dim=['lat', 'lon'])` collapses the spatial dimensions — like `apply(arr, 1, mean)` across lat/lon in R.

`.compute()` is the step that **actually downloads the data** from AWS S3. This may take a minute or two per dataset.

In [ ]:
print("Downloading data from AWS S3 — this may take a few minutes...")

# Compute spatial mean and download data
# .compute() triggers the actual S3 download
tas_hist   = gl_hist['tas'].mean(dim=['lat', 'lon']).compute()
tas_ssp245 = gl_ssp245['tas'].mean(dim=['lat', 'lon']).compute()
tas_ssp585 = gl_ssp585['tas'].mean(dim=['lat', 'lon']).compute()

# Convert from Kelvin to Celsius (standard in climate science)
tas_hist_c   = tas_hist   - 273.15
tas_ssp245_c = tas_ssp245 - 273.15
tas_ssp585_c = tas_ssp585 - 273.15

print("Done!")
print(f"Historical temperature range: {float(tas_hist_c.min()):.1f} to {float(tas_hist_c.max()):.1f} °C")

## 9. Plot a Time Series

Monthly data is noisy. A 5-year (60-month) rolling average smooths the signal so long-term trends are easier to see — equivalent to `zoo::rollmean()` in R.

We convert to `pandas.Series` to use its built-in `.rolling()` method, then plot with `matplotlib`.

In [ ]:
window = 60  # 60 months = 5-year rolling average

# Convert xarray DataArrays to pandas Series for easy rolling mean
# (like zoo::rollmean in R, but center=True aligns the window to the center)
s_hist   = pd.Series(tas_hist_c.values,   index=pd.DatetimeIndex(tas_hist_c.time.values))
s_ssp245 = pd.Series(tas_ssp245_c.values, index=pd.DatetimeIndex(tas_ssp245_c.time.values))
s_ssp585 = pd.Series(tas_ssp585_c.values, index=pd.DatetimeIndex(tas_ssp585_c.time.values))

hist_smooth   = s_hist.rolling(window, center=True).mean()
ssp245_smooth = s_ssp245.rolling(window, center=True).mean()
ssp585_smooth = s_ssp585.rolling(window, center=True).mean()

# Plot — fig/ax is matplotlib's way of creating a figure (like ggplot() in R)
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(hist_smooth,   color='gray',       linewidth=2, label='Historical')
ax.plot(ssp245_smooth, color='steelblue',  linewidth=2, label='SSP2-4.5 (moderate emissions)')
ax.plot(ssp585_smooth, color='firebrick',  linewidth=2, label='SSP5-8.5 (high emissions)')

# Vertical line marking where historical ends and projections begin
ax.axvline(pd.Timestamp('2015-01-01'), color='black', linestyle='--',
           alpha=0.5, label='2015: historical → projection')

ax.set_xlabel('Year')
ax.set_ylabel('Temperature (°C)')
ax.set_title('CESM2: Great Lakes Region — 5-Year Rolling Mean Temperature')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## Next Steps

Now that you can access, process, and plot CMIP6 data, here are some ways to extend this:

- **Different variable:** replace `'tas'` with `'pr'` (precipitation) or `'psl'` (sea-level pressure) in the catalog search
- **Different model:** replace `'CESM2'` with `'CanESM5'` (Canadian) or `'MIROC6'` (Japanese) to compare model responses
- **Different region:** update `lat_min/max` and `lon_min/max` to any region you care about
- **Multiple models on one plot:** loop over a list of model names and overlay all results

A good next resource is the [Pangeo CMIP6 cookbook](https://projectpythia.org/cmip6-cookbook/README.html), which covers more advanced analysis patterns.